# 00.00_plot_figures

测序 reads/genes 饱和曲线与 DNB 模块统计绘图。

- 当前文件：`analysis/00_figures/00.00_plot_figures.ipynb`
- 原始来源：`Codes/00.00_plot_figures.ipynb`（旧编号仅用于溯源）。
- 运行内核：**python**。
- 导入依赖：`io`, `matplotlib.pyplot`, `numpy`, `os`, `pandas`, `scipy.optimize`, `seaborn`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


In [ ]:
fig_dir = '/share/home/zhangze/zz/NeuralOrigin/Figures'

ZZ：01三个批次的Reads和Genes关系曲线

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ----------------------------
# Data
# ----------------------------
batch1_x = np.array([0, 654, 1307, 2614, 3921, 5228, 6535, 7842, 9149, 10456, 11763, 13070], dtype=float)
batch1_y = np.array([0, 188, 285, 400, 471, 521, 558, 588, 613, 634, 651, 665], dtype=float)

batch2_x = np.array([0, 575, 1149, 2298, 3448, 4597, 5746, 6895, 8045, 9194, 10343, 11492], dtype=float)
batch2_y = np.array([0, 176, 272, 387, 461, 513, 553, 586, 611, 634, 652, 669], dtype=float)

batch3_x = np.array([0, 454, 908, 1816, 2724, 3633, 4541, 5449, 6357, 7265, 8173, 9081], dtype=float)
batch3_y = np.array([0, 152, 242, 355, 429, 484, 526, 560, 587, 611, 632, 650], dtype=float)

# ----------------------------
# Saturation fit + R^2
# ----------------------------
def sat_model(x, vmax, k):
    # Michaelis–Menten / 1-site saturation
    return (vmax * x) / (k + x + 1e-12)

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

def fit_and_r2(x, y):
    k0 = float(np.median(x[x > 0])) if np.any(x > 0) else 1.0
    p0 = [float(np.max(y)), k0]
    popt, _ = curve_fit(sat_model, x, y, p0=p0, maxfev=10000)
    y_fit = sat_model(x, *popt)
    return popt, r2_score(y, y_fit)

p1, r2_1 = fit_and_r2(batch1_x, batch1_y)
p2, r2_2 = fit_and_r2(batch2_x, batch2_y)
p3, r2_3 = fit_and_r2(batch3_x, batch3_y)

# ----------------------------
# Plot
# ----------------------------
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)  # 8:6

ax.plot(batch1_x, batch1_y, marker='o', linewidth=2, color="#EB696C",
        label=f"Batch_au1 (R²={r2_1:.3f})")
ax.plot(batch2_x, batch2_y, marker='o', linewidth=2, color="#0FD73D",
        label=f"Batch_au2 (R²={r2_2:.3f})")
ax.plot(batch3_x, batch3_y, marker='o', linewidth=2, color="#669BF8",
        label=f"Batch_au3 (R²={r2_3:.3f})")

ax.set_xlabel("Mean Reads per Cell")
ax.set_ylabel("Median Genes per Cell")
ax.grid(False)

# full rectangular border (all spines visible)
for side in ["top", "right", "bottom", "left"]:
    ax.spines[side].set_visible(True)
    ax.spines[side].set_linewidth(1.0)

ax.legend(frameon=False)  # legend显示R²；不加外框更接近常见论文风格

plt.tight_layout()

# Save as PDF (dpi对PDF影响不大，但保留你的dpi设置)
fig.savefig(fig_dir + '/01.Curve.genes_VS_reads_with_R2.pdf', format="pdf", bbox_inches="tight")
fig.savefig(fig_dir + '/01.Curve.genes_VS_reads_with_R2.png', format="png", bbox_inches="tight")

plt.show()


In [ ]:
import numpy as np
from scipy.optimize import curve_fit

def sat_model(x, vmax, k):
    # Michaelis–Menten / saturation curve
    return (vmax * x) / (k + x + 1e-12)

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

batches = {
    "batch1": (
        np.array([0, 654, 1307, 2614, 3921, 5228, 6535, 7842, 9149, 10456, 11763, 13070], dtype=float),
        np.array([0, 188, 285, 400, 471, 521, 558, 588, 613, 634, 651, 665], dtype=float),
    ),
    "batch2": (
        np.array([0, 575, 1149, 2298, 3448, 4597, 5746, 6895, 8045, 9194, 10343, 11492], dtype=float),
        np.array([0, 176, 272, 387, 461, 513, 553, 586, 611, 634, 652, 669], dtype=float),
    ),
    "batch3": (
        np.array([0, 454, 908, 1816, 2724, 3633, 4541, 5449, 6357, 7265, 8173, 9081], dtype=float),
        np.array([0, 152, 242, 355, 429, 484, 526, 560, 587, 611, 632, 650], dtype=float),
    ),
}

for name, (x, y) in batches.items():
    # initial guess: vmax ~ max(y), k ~ median positive x
    k0 = float(np.median(x[x > 0])) if np.any(x > 0) else 1.0
    p0 = [float(np.max(y)), k0]

    popt, _ = curve_fit(sat_model, x, y, p0=p0, maxfev=10000)
    y_fit = sat_model(x, *popt)
    r2 = r2_score(y, y_fit)

    print(f"{name} R^2 = {r2:.6f} (vmax={popt[0]:.3f}, k={popt[1]:.3f})")


ZZ：81DNB模块统计

In [ ]:
import pandas as pd
import io

data = """rank,rank_all,bestMODULE,SCORE,genes,MEAN,SD,CV,PCC_IN,PCC_OUT,resource
1,1,TRUE,47.4543872981175,"OG0002612,OG0005723,OG0002450,OG0002725,OG0002275,OG0000156,OG0003606,OG0000086,OG0000328,OG0000053,OG0000203,OG0005351,OG0001683,OG0002242,OG0002112,OG0000290,OG0000426",0.000815753532362361,1.45797953765505,9.11960755535012,0.238446849300297,0.0302058531982091,"Placozoa_74"
2,2,TRUE,46.3065071587403,"OG0004646,OG0004629,OG0003941,OG0003379,OG0000247,OG0004010,OG0004250,OG0002876,OG0004075,OG0002801,OG0001449,OG0003804,OG0002977,OG0000540,OG0003048,OG0003847,OG0003886,OG0004895,OG0003828,OG0003773,OG0004369,OG0003326,OG0001229,OG0004413,OG0003333,OG0004490,OG0003148,OG0005961,OG0003634,OG0003769,OG0004015,OG0004502,OG0002488,OG0004812,OG0003494,OG0003465,OG0003829,OG0003702,OG0004392,OG0004295,OG0000982,OG0004002,OG0003043,OG0003154,OG0000863,OG0002707",-0.680924255787413,1.33580865938499,-2.04613531812393,0.144503516238321,0.028272197197465,"Placozoa_236"
3,3,TRUE,44.8783220761003,"OG0000442,OG0001096,OG0003248,OG0000331,OG0001937,OG0000586,OG0001002,OG0000327,OG0000164,OG0000133,OG0002929,OG0000429,OG0001901,OG0000081,OG0000082,OG0000016",0.664344348472431,1.82234964664444,4.10863564496166,0.163547582819714,0.0265643514261223,"Placozoa_194"
4,6,TRUE,35.2920545754233,"OG0002494,OG0000237,OG0006575,OG0000132,OG0000020,OG0002579,OG0000714,OG0000260,OG0000700,OG0000128,OG0000554,OG0001150,OG0000225,OG0001036,OG0004510,OG0000036,OG0000550,OG0001644,OG0000371,OG0000072,OG0000976,OG0002183",0.526451715209257,1.4626350626058,4.04522058534844,0.151807023692806,0.0295095267107864,"Placozoa_185"
5,7,TRUE,28.2395994209513,"OG0005336,OG0003238,OG0005376,OG0000483,OG0003240,OG0000332,OG0003481,OG0000799",0.300607317032485,1.42984928261759,7.18778823537975,0.157924335861206,0.0226165200846595,"Placozoa_160"
6,8,TRUE,24.5049762287833,"OG0004979,OG0000928,OG0006748,OG0002729,OG0000112,OG0004887,OG0001352,OG0000350,OG0001312",0.339606911504082,1.49460968185931,3.85068028034776,0.150253991099951,0.0274929142235459,"Placozoa_207"
7,11,TRUE,22.3779174014658,"OG0005800,OG0004822,OG0004364,OG0001444,OG0000166,OG0000723,OG0004632,OG0001145,OG0000031,OG0002197",-0.144065988315307,1.20606352349381,6.54892360540429,0.128092026784684,0.0218309489327834,"Placozoa_64"
8,12,TRUE,20.3799846345523,"OG0000403,OG0000722,OG0001451,OG0001067,OG0000058,OG0000013,OG0003295,OG0000003,OG0000348,OG0003625,OG0000424",0.502489651085572,1.44928102974046,6.2566269129094,0.0973796255095316,0.0229674721261779,"Placozoa_140"
9,15,TRUE,17.7213353663668,"OG0001732,OG0000480,OG0000115,OG0003226,OG0004373,OG0001933,OG0005883,OG0000612,OG0000245,OG0000238,OG0000190,OG0000025,OG0000038,OG0001204,OG0000139,OG0000704,OG0003321,OG0000171",-0.551743609035913,1.059550752177,-2.22795432845294,0.114049551638119,0.0289304477062409,"Placozoa_225"
10,21,TRUE,13.6414755707795,"OG0005194,OG0004955,OG0006660,OG0002227,OG0003269,OG0001402,OG0001023,OG0006706",-0.16891925232038,0.48975084752628,-3.19938853481094,0.156126509554933,0.0158538772245597,"Placozoa_241"
"""

df = pd.read_csv(io.StringIO(data))
print(df.info())
print(df.head())

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

# ==================== 配置区域 ====================
# 请确保该目录已存在，或者使用下面两行代码自动创建
# fig_dir = "./output_figures"
# os.makedirs(fig_dir, exist_ok=True)
# ==================================================

# Calculate gene count for each row
df['gene_count'] = df['genes'].apply(lambda x: len(x.split(',')))

# Set up matplotlib style
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']  # Support Chinese if needed, or fallback
plt.rcParams['axes.unicode_minus'] = False


# ------------------ Plot : Scatter plot of PCC_IN vs PCC_OUT ------------------
fig2, ax2 = plt.subplots(figsize=(6, 8))

scatter = ax2.scatter(df['PCC_OUT'], df['PCC_IN'], c=df['SCORE'], s=df['gene_count']*10, cmap='coolwarm', alpha=0.8, edgecolors='w')
fig2.colorbar(scatter, ax=ax2, label='SCORE')

ax2.set_title('PCC_IN vs PCC_OUT (Size represents Gene Count)', fontsize=14)
ax2.set_xlabel('PCC_OUT (Pearson Correlation Outside)', fontsize=12)
ax2.set_ylabel('PCC_IN (Pearson Correlation Inside)', fontsize=12)

# Add annotations for resource names
for i, row in df.iterrows():
    ax2.annotate(row['resource'], (row['PCC_OUT'], row['PCC_IN']), textcoords="offset points", xytext=(5,5), ha='left', fontsize=9)

fig2.tight_layout()
# 双格式保存
fig2.savefig(fig_dir + '/81.Scatter.pcc_correlation.pdf', format="pdf", bbox_inches="tight")
fig2.savefig(fig_dir + '/81.Scatter.pcc_correlation.png', format="png", bbox_inches="tight", dpi=300)
plt.show()


# ==================== 打印日志 ====================
print("Plots generated and saved successfully!")
print(df[['resource', 'SCORE', 'gene_count', 'PCC_IN', 'PCC_OUT']].to_string())

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

# ==================== 配置区域 ====================
# 请确保该目录已存在，或者使用下面两行代码自动创建
# fig_dir = "./output_figures"
# os.makedirs(fig_dir, exist_ok=True)
# ==================================================

# 1. 计算基因数量
df['gene_count'] = df['genes'].apply(lambda x: len(x.split(',')))

# 2. 核心修改：根据 rank 动态重命名模块为 DNB Module_X
df = df.sort_values(by='rank').reset_index(drop=True)
df['resource'] = [f"Module_{i+1}" for i in range(len(df))]

# Set up matplotlib style
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']  # Support Chinese if needed, or fallback
plt.rcParams['axes.unicode_minus'] = False


# ------------------ Plot : Scatter plot of PCC_IN vs PCC_OUT ------------------
fig2, ax2 = plt.subplots(figsize=(7, 8))

# 绘制气泡散点图
scatter = ax2.scatter(
    df['PCC_OUT'], 
    df['PCC_IN'], 
    c=df['SCORE'], 
    s=df['gene_count'] * 10, 
    cmap='coolwarm', 
    alpha=0.8, 
    edgecolors='w'
)

# 1. 添加颜色的图例 (Colorbar)
fig2.colorbar(scatter, ax=ax2, label='SCORE')

# 2. 自动提取并添加气泡大小图例 (Gene Count Legend)
handles, labels = scatter.legend_elements(prop="sizes", alpha=0.6, num=4)

true_labels = []
for label in labels:
    text_str = label.get_text() if hasattr(label, 'get_text') else str(label)
    clean_num = text_str.replace('$\\mathdefault{', '').replace('}$', '')
    true_labels.append(str(int(float(clean_num) / 10)))

# 将 Gene Count 图例固定在图形右上角
size_legend = ax2.legend(
    handles, 
    true_labels, 
    loc="upper right", 
    title="Gene Count",
    title_fontsize=11,
    fontsize=9,
    frameon=True
)
ax2.add_artist(size_legend)

# 设置标题与坐标轴
ax2.set_title('PCC_IN vs PCC_OUT (Color: SCORE, Size: Gene Count)', fontsize=14)
ax2.set_xlabel('PCC_OUT (Pearson Correlation Outside)', fontsize=12)
ax2.set_ylabel('PCC_IN (Pearson Correlation Inside)', fontsize=12)

# 修改此处：图面上的标签也会自动更新为新名字 DNB Module_X
for i, row in df.iterrows():
    ax2.annotate(row['resource'], (row['PCC_OUT'], row['PCC_IN']), textcoords="offset points", xytext=(5,5), ha='left', fontsize=9)

fig2.tight_layout()
# 双格式保存
fig2.savefig(fig_dir + '/81.Scatter.pcc_correlation.pdf', format="pdf", bbox_inches="tight")
fig2.savefig(fig_dir + '/81.Scatter.pcc_correlation.png', format="png", bbox_inches="tight", dpi=300)
plt.show()


# ==================== 打印日志 ====================
print("Plots generated and saved successfully!")
print(df[['resource', 'SCORE', 'gene_count', 'PCC_IN', 'PCC_OUT']].to_string())